# 🎬 Advanced AI Movie Generator — Wan 2.1 (Google Colab)

**Goal:** One project prompt → multiple short AI-generated scenes → one stitched MP4 (60–120 seconds).

### What this notebook does
- Wan 2.1 Text-to-Video 1.3B generates short 480p clips.
- You provide a list of scene prompts (one per line) to control the story and visual continuity.
- The notebook calculates how many clips are needed for the selected target duration.
- It generates clips, stitches them into one MP4, and provides a download link.
- Style presets: realistic, cinematic, 3D animation, anime, cartoon, and Shorts.

### Important limitations
- This is an inference workflow using pretrained weights, not training a new foundation model.
- Wan 2.1 1.3B produces short clips, not native 1–2 minute videos. This notebook joins many short clips.
- A 60-second video needs about 20 clips; a 120-second video needs about 40 clips at ~3 seconds each. This can take hours and may exceed free Colab GPU quotas.
- Output is silent. Add licensed music/voice-over in an editor, or extend this notebook with a separate audio workflow.
- The quality and consistency depend on prompts, seed, GPU, and model limitations. No guaranteed professional/film-studio result.

**Recommended:** Colab → Runtime → Change runtime type → GPU. A high-memory GPU is strongly preferred.

In [ ]:
# 1) Install dependencies
!pip -q install -U diffusers transformers accelerate ftfy gradio imageio imageio-ffmpeg

In [ ]:
# 2) Check GPU and storage
import torch, os, gc, subprocess, json, math, uuid, time
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("GPU nahi mila. Colab: Runtime > Change runtime type > GPU select karein.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Free disk (GB):", round(subprocess.check_output(["bash","-lc","df -BG /content | tail -1 | awk '{print $4}'"]).decode().strip().replace("G","") and float(subprocess.check_output(["bash","-lc","df -BG /content | tail -1 | awk '{print $4}'"]).decode().strip().replace("G","")), 1))

In [ ]:
# 3) Load Wan 2.1 text-to-video model
# First run downloads several GB of model weights.
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
major, minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID, subfolder="vae", torch_dtype=torch.float32
)
pipe = WanPipeline.from_pretrained(
    MODEL_ID, vae=vae, torch_dtype=DTYPE
)
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config, flow_shift=3.0
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

print("Wan 2.1 is ready. GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 4) Style presets and generation settings
STYLE_PRESETS = {
    "Realistic": "photorealistic, natural motion, realistic textures, physically plausible lighting",
    "Professional cinematic": "high-end cinematic film look, deliberate composition, dramatic lighting, smooth camera movement, rich detail",
    "3D animation": "high-quality 3D animated film, detailed 3D characters and environments, expressive motion, polished render",
    "Anime": "beautiful anime film aesthetic, expressive animation, detailed backgrounds, cinematic composition",
    "Cartoon": "polished colorful cartoon animation, appealing shapes, expressive characters, fluid motion",
    "YouTube Shorts": "vertical social video aesthetic, bold visual storytelling, clear subject, fast engaging composition"
}

NEGATIVE_PROMPT = (
    "low quality, blurry, flicker, jitter, warped objects, distorted faces, "
    "deformed anatomy, extra limbs, inconsistent character, bad hands, text, "
    "subtitles, watermark, logo, abrupt camera movement"
)

OUTPUT_DIR = Path("/content/advanced_video_project")
CLIPS_DIR = OUTPUT_DIR / "clips"
CLIPS_DIR.mkdir(parents=True, exist_ok=True)

# Wan 2.1 T2V 1.3B: 49 frames at 16 fps is approximately 3 seconds.
FPS = 16
FRAMES_PER_CLIP = 49
SECONDS_PER_CLIP = FRAMES_PER_CLIP / FPS
WIDTH, HEIGHT = 832, 480

print("Approx seconds per clip:", SECONDS_PER_CLIP)
print("Style presets:", ", ".join(STYLE_PRESETS.keys()))

In [ ]:
# 5) Scene generation + final video assembly
import re, shutil

def clean_scene_lines(scene_text):
    lines = [re.sub(r"^\s*(?:\d+[.)-]|[-*])\s*", "", x).strip()
             for x in (scene_text or "").splitlines()]
    return [x for x in lines if x]

def make_scene_prompt(project_prompt, scene_prompt, style_name, scene_number, total_scenes):
    style = STYLE_PRESETS[style_name]
    return (
        f"{project_prompt.strip()}. "
        f"Scene {scene_number} of {total_scenes}: {scene_prompt}. "
        f"Visual style: {style}. "
        "Maintain coherent visual storytelling, natural movement, consistent visual world, "
        "clear subject, cinematic composition."
    )

def generate_one_clip(prompt, seed, steps, clip_path):
    generator = torch.Generator(device="cuda").manual_seed(int(seed))
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=HEIGHT,
        width=WIDTH,
        num_frames=FRAMES_PER_CLIP,
        num_inference_steps=int(steps),
        guidance_scale=5.0,
        generator=generator
    )
    export_to_video(result.frames[0], str(clip_path), fps=FPS)
    del result
    gc.collect()
    torch.cuda.empty_cache()
    return str(clip_path)

def stitch_clips(clip_paths, output_path):
    # Re-encode all clips to a consistent MP4 format before concatenating.
    list_file = OUTPUT_DIR / "concat_list.txt"
    normalized = []
    for i, src in enumerate(clip_paths):
        dst = OUTPUT_DIR / f"normalized_{i:03d}.mp4"
        cmd = [
            "ffmpeg", "-y", "-i", str(src),
            "-vf", f"scale={WIDTH}:{HEIGHT}:force_original_aspect_ratio=decrease,pad={WIDTH}:{HEIGHT}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={FPS}",
            "-an", "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
            "-pix_fmt", "yuv420p", str(dst)
        ]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        normalized.append(dst)
    with open(list_file, "w") as f:
        for p in normalized:
            f.write(f"file '{p.as_posix()}'\n")
    subprocess.run([
        "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_file),
        "-c", "copy", "-movflags", "+faststart", str(output_path)
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    return str(output_path)

def create_movie(project_prompt, scene_text, target_seconds, style_name,
                 steps=25, base_seed=1234, max_scenes=40):
    project_prompt = (project_prompt or "").strip()
    scenes = clean_scene_lines(scene_text)
    if len(project_prompt) < 10:
        raise ValueError("Project prompt ko thoda detail mein likhein.")
    if not scenes:
        raise ValueError("Har scene ka prompt alag line mein likhein.")
    target_seconds = int(target_seconds)
    if target_seconds not in (60, 90, 120):
        raise ValueError("Duration 60, 90 ya 120 seconds select karein.")

    needed = math.ceil(target_seconds / SECONDS_PER_CLIP)
    if needed > max_scenes:
        raise ValueError(f"Is duration ke liye {needed} clips chahiye. Current safety limit {max_scenes} hai.")

    # Cycle through scene descriptions if fewer descriptions than required clips.
    expanded_scenes = [scenes[i % len(scenes)] for i in range(needed)]
    project_id = uuid.uuid4().hex[:8]
    project_folder = OUTPUT_DIR / project_id
    project_folder.mkdir(parents=True, exist_ok=True)

    clip_paths = []
    status_lines = [f"Creating {needed} clips (~{target_seconds} seconds total)."]
    for idx, scene in enumerate(expanded_scenes):
        prompt = make_scene_prompt(project_prompt, scene, style_name, idx+1, needed)
        clip_path = project_folder / f"scene_{idx+1:03d}.mp4"
        status_lines.append(f"Generating scene {idx+1}/{needed}...")
        generate_one_clip(prompt, int(base_seed)+idx, int(steps), clip_path)
        clip_paths.append(clip_path)

    final_path = project_folder / "final_movie.mp4"
    stitch_clips(clip_paths, final_path)
    actual = needed * SECONDS_PER_CLIP
    return str(final_path), "\n".join(status_lines + [f"Done. Approx duration: {actual:.1f} seconds."])

print("Movie pipeline ready.")

## 6) Launch the video studio

Write one overall project description, then provide scene prompts on separate lines. If you provide fewer scenes than needed, the notebook cycles through them; for best storytelling, write a distinct scene description for each clip.

**Example project prompt:** `A young explorer discovers an ancient hidden city beneath the desert`

**Example scene lines:**
1. `Wide aerial shot of a vast golden desert at sunrise`
2. `The explorer walks toward a half-buried stone doorway`
3. `Close-up as the explorer brushes sand from mysterious carvings`
4. `The stone doorway slowly opens, revealing blue light`
5. `A huge underground city is revealed, glowing with ancient lights`

For a 60-second video, aim to write around 20 scene lines. For 120 seconds, around 40.

In [ ]:
# 7) Gradio UI
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft(), title="Advanced AI Movie Generator") as app:
    gr.Markdown("# 🎬 Advanced AI Movie Generator")
    gr.Markdown("Prompt-driven multi-scene video creation with Wan 2.1")
    project = gr.Textbox(
        label="Overall video idea / story",
        lines=2,
        placeholder="A brave astronaut explores an abandoned space station..."
    )
    scenes = gr.Textbox(
        label="Scene prompts — one scene per line",
        lines=10,
        placeholder="Wide shot of a spaceship approaching a silent space station\nThe astronaut enters the dark corridor\nA mysterious light appears at the end of the hall..."
    )
    with gr.Row():
        duration = gr.Radio([60, 90, 120], value=60, label="Final video duration (seconds)")
        style = gr.Dropdown(list(STYLE_PRESETS.keys()), value="Professional cinematic", label="Visual style")
    with gr.Row():
        steps = gr.Slider(15, 35, value=25, step=5, label="Inference steps (higher = slower)")
        seed = gr.Number(value=1234, precision=0, label="Base seed")
    run_btn = gr.Button("🎥 Generate full video", variant="primary")
    output_video = gr.Video(label="Final MP4 movie")
    status = gr.Textbox(label="Generation log", lines=6)
    run_btn.click(
        fn=create_movie,
        inputs=[project, scenes, duration, style, steps, seed],
        outputs=[output_video, status]
    )

app.queue(max_size=1).launch(share=True, debug=True)

## 8) Optional: Download final MP4 manually

After generation, the Gradio video player provides a download control. The MP4 is also saved under `/content/advanced_video_project/`.

If the Colab runtime disconnects, files in `/content` may be lost. Download your final MP4 before closing the session.

### Troubleshooting
- **CUDA out of memory:** restart runtime, use a GPU with more VRAM, close other GPU processes. This notebook already uses CPU offload and VAE tiling.
- **Generation takes too long:** try 15 inference steps for a test. A 1–2 minute project still requires many separate generations.
- **Runtime disconnects:** generate shorter 60-second projects or run on a paid/high-memory GPU environment.
- **Visual continuity:** repeat the same character description, clothing, environment, and visual details in every scene prompt. Wan T2V does not guarantee exact character consistency across clips.
- **Audio:** final output is silent; add narration/music separately using properly licensed audio.
- **Style:** presets guide the prompt but do not guarantee a perfect style transformation.